In [1]:

import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import joblib

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:

today = datetime.now(ZoneInfo("Asia/Kolkata")).date()

end_date = today - timedelta(days=1)
start_date = end_date - timedelta(days=729)

url = "https://air-quality-api.open-meteo.com/v1/air-quality"

params = {
    "latitude": 28.6139,
    "longitude": 77.2090,
    "start_date": start_date.isoformat(),
    "end_date": end_date.isoformat(),
    "hourly": "us_aqi",
    "timezone": "Asia/Kolkata"
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

api_data = response.json()

hourly_data = pd.DataFrame(api_data["hourly"])
hourly_data["time"] = pd.to_datetime(hourly_data["time"])
hourly_data["Date"] = hourly_data["time"].dt.date

daily = hourly_data.groupby("Date").agg(
    AQI=("us_aqi", "mean"),
    Count=("us_aqi", "count")
).reset_index()

daily = daily[daily["Count"] == 24]
daily = daily[["Date", "AQI"]].copy()
daily["Date"] = pd.to_datetime(daily["Date"])

print("Total complete days:", len(daily))
print("First date:", daily["Date"].min())
print("Last date:", daily["Date"].max())

display(daily.head())
display(daily.tail())


Total complete days: 730
First date: 2024-11-19 00:00:00
Last date: 2026-11-18 00:00:00


,Date,AQI
0,2024-11-19,178.625000
1,2024-11-20,181.500000
2,2024-11-21,190.750000
3,2024-11-22,203.208333
4,2024-11-23,246.166667


,Date,AQI
725,2026-11-14,198.958333
726,2026-11-15,162.958333
727,2026-11-16,173.958333
728,2026-11-17,157.041667
729,2026-11-18,165.208333


In [3]:

live_df = daily.copy()

live_df = live_df.sort_values("Date")
live_df = live_df.set_index("Date").asfreq("D")

live_df["Month"] = live_df.index.month
live_df["Day"] = live_df.index.day
live_df["DayOfWeek"] = live_df.index.dayofweek

live_df["AQI_Lag1"] = live_df["AQI"].shift(1)
live_df["AQI_Lag3"] = live_df["AQI"].shift(3)
live_df["AQI_Lag7"] = live_df["AQI"].shift(7)

live_df["AQI_Rolling3"] = live_df["AQI"].shift(1).rolling(3).mean()
live_df["AQI_Rolling7"] = live_df["AQI"].shift(1).rolling(7).mean()

for h in [1, 3, 7]:
    live_df[f"Target_{h}Day"] = live_df["AQI"].shift(-h)

live_df = live_df.dropna().copy()

print("Final dataset shape:", live_df.shape)
display(live_df.head())


Final dataset shape: (716, 12)


,AQI,Month,Day,DayOfWeek,AQI_Lag1,AQI_Lag3,AQI_Lag7,AQI_Rolling3,AQI_Rolling7,Target_1Day,Target_3Day,Target_7Day
Date,,,,,,,,,,,,
2024-11-26,177.791667,11,26,1,171.583333,246.166667,178.625000,210.972222,198.142857,163.750000,189.541667,153.750000
2024-11-27,163.750000,11,27,2,177.791667,215.166667,181.500000,188.180556,198.023810,177.583333,193.291667,112.333333
2024-11-28,177.583333,11,28,3,163.750000,171.583333,190.750000,171.041667,195.488095,189.541667,165.333333,95.416667
2024-11-29,189.541667,11,29,4,177.583333,177.791667,203.208333,173.041667,193.607143,193.291667,159.458333,110.875000
2024-11-30,193.291667,11,30,5,189.541667,163.750000,246.166667,176.958333,191.654762,165.333333,153.750000,156.250000


In [4]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

live_features = [
    "AQI", "Month", "Day", "DayOfWeek",
    "AQI_Lag1", "AQI_Lag3", "AQI_Lag7",
    "AQI_Rolling3", "AQI_Rolling7"
]

split = int(len(live_df) * 0.8)

live_train = live_df.iloc[:split - 7]
live_test = live_df.iloc[split:]

live_models = {}
live_results = []

for h in [1, 3, 7]:
    target = f"Target_{h}Day"

    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )

    model.fit(
        live_train[live_features],
        live_train[target]
    )

    y_pred = model.predict(live_test[live_features])
    y_true = live_test[target]

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    live_models[h] = model

    live_results.append({
        "Horizon": f"{h} Day",
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "R2": round(r2, 3)
    })

live_results_df = pd.DataFrame(live_results)

display(live_results_df)


,Horizon,MAE,RMSE,R2
0,1 Day,40.11,68.18,0.489
1,3 Day,63.51,101.68,-0.226
2,7 Day,65.98,102.23,-0.243


In [5]:

joblib.dump({
    "models": live_models,
    "features": live_features
}, "Live_AQI_Models.pkl")

live_results_df.to_csv(
    "Live_AQI_Results.csv",
    index=False
)

print("Models saved successfully!")


Models saved successfully!


In [6]:

baseline_results = []

for h in [1, 3, 7]:
    target = f"Target_{h}Day"

    y_true = live_test[target]
    y_pred = live_test["AQI"]

    baseline_results.append({
        "Horizon": f"{h} Day",
        "Baseline_MAE": round(
            mean_absolute_error(y_true, y_pred), 2
        ),
        "Model_MAE": round(
            live_results_df.loc[
                live_results_df["Horizon"] == f"{h} Day",
                "MAE"
            ].iloc[0], 2
        )
    })

baseline_comparison = pd.DataFrame(baseline_results)

baseline_comparison["Improvement_%"] = (
    (baseline_comparison["Baseline_MAE"] -
     baseline_comparison["Model_MAE"]) /
    baseline_comparison["Baseline_MAE"] * 100
).round(2)

display(baseline_comparison)


,Horizon,Baseline_MAE,Model_MAE,Improvement_%
0,1 Day,37.48,40.11,-7.02
1,3 Day,69.66,63.51,8.83
2,7 Day,74.12,65.98,10.98


In [7]:

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor

new_models = {
    "Linear Regression": LinearRegression(),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        random_state=42
    )
}

comparison_results = []
candidate_models = {}

for h in [1, 3, 7]:
    target = f"Target_{h}Day"

    for name, model in new_models.items():
        model.fit(
            live_train[live_features],
            live_train[target]
        )

        y_pred = model.predict(live_test[live_features])
        y_true = live_test[target]

        comparison_results.append({
            "Horizon": f"{h} Day",
            "Model": name,
            "MAE": round(mean_absolute_error(y_true, y_pred), 2),
            "RMSE": round(
                np.sqrt(mean_squared_error(y_true, y_pred)), 2
            ),
            "R2": round(r2_score(y_true, y_pred), 3)
        })

        candidate_models[(h, name)] = model

comparison_df = pd.DataFrame(comparison_results)

display(comparison_df)


,Horizon,Model,MAE,RMSE,R2
0,1 Day,Linear Regression,34.92,64.42,0.543
1,1 Day,Gradient Boosting,36.00,67.04,0.505
2,3 Day,Linear Regression,58.99,95.33,-0.078
3,3 Day,Gradient Boosting,60.46,99.26,-0.169
4,7 Day,Linear Regression,64.53,96.13,-0.099
5,7 Day,Gradient Boosting,60.39,100.15,-0.193


In [8]:

selected_models = {
    1: candidate_models[(1, "Linear Regression")],
    3: candidate_models[(3, "Linear Regression")],
    7: candidate_models[(7, "Gradient Boosting")]
}

joblib.dump({
    "models": selected_models,
    "features": live_features,
    "aqi_type": "US AQI",
    "location": "Delhi"
}, "Live_AQI_Models.pkl")

print("Selected models saved successfully!")


Selected models saved successfully!
